In [14]:
using Pkg
Pkg.add(["CSV", "DataFrames", "Dates", "Statistics", "Flux", "MLUtils", "Clustering", "Random", "OneHotArrays", "LinearAlgebra", "CategoricalArrays", "Printf"])

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [15]:
using CSV, DataFrames, Dates, Statistics, Random, LinearAlgebra
using Flux, OneHotArrays, Clustering, MLUtils, CategoricalArrays
using Flux: @epochs
using Printf

# --- 1. Custom Focal Loss per il Bilanciamento ---
# Alpha: pesi per classe, Gamma: fattore di focalizzazione sui campioni difficili
# Funzione Loss Corretta e Robusta
function focal_loss(y_pred, y_true; gamma=2.0f0, alpha=nothing)
    # Usiamo 1f-7 che è la notazione scientifica corretta per Float32
    epsilon = 1f-7 
    one_f32 = 1f0 # 1.0 in Float32 esplicito
    
    # Clamp per evitare log(0)
    y_pred = clamp.(y_pred, epsilon, one_f32 - epsilon)
    
    # Cross entropy term
    ce_loss = -y_true .* log.(y_pred)
    
    # Focal term: (1 - p_t)^gamma
    weight = (one_f32 .- y_pred) .^ gamma
    
    loss = weight .* ce_loss
    
    if !isnothing(alpha)
        loss = loss .* alpha
    end
    
    return mean(sum(loss, dims=1))
end

# --- 2. Feature Engineering Avanzato ---

# Cyclic Encoding per ore/mesi
function encode_cyclic(data, col, max_val)
    sin_feat = sin.(2π .* data[!, col] ./ max_val)
    cos_feat = cos.(2π .* data[!, col] ./ max_val)
    return sin_feat, cos_feat
end

# Velocity Feature: Transazioni per IP nell'ultima ora
function calculate_velocity(df)
    # Assumiamo che df sia ordinato per data o lo ordiniamo internamente
    # Nota: Per performance su grandi dataset, questo dovrebbe essere ottimizzato
    # Qui usiamo un approccio vettoriale semplificato per la demo
    
    # Creiamo un dizionario per contare le frequenze recenti
    velocities = zeros(Float32, nrow(df))
    # Simulazione velocity: raggruppamento per IP e conteggio
    # In produzione si farebbe con finestre temporali scorrevoli
    gdf = groupby(df, Symbol("IP Address"))
    
    # Normalizziamo semplicemente per frequenza relativa nel dataset (Proxy di velocity)
    # Dato che simulare una window function temporale esatta è lento in puro DF row-loop
    transform!(gdf, nrow => :IP_Velocity)
    return df[!, :IP_Velocity]
end

# Mismatch Feature
function check_address_mismatch(df)
    # 1 se Billing != Shipping, 0 altrimenti
    return df[!, Symbol("Billing Address")] .!= df[!, Symbol("Shipping Address")]
end

check_address_mismatch (generic function with 1 method)

In [16]:
# Struct per salvare lo stato del training (medie, deviazioni, cluster centers)
mutable struct Preprocessor
    means::Dict{Symbol, Float64}
    stds::Dict{Symbol, Float64}
    cat_maps::Dict{Symbol, Dict{String, Int}} # One-hot maps
    cluster_model::Any # KMeans result
    cluster_threshold::Float64 # Soglia per definire "Sospetto"
    fraud_ratio::Vector{Float32} # Per calcolo alpha focal loss
end

function Preprocessor()
    return Preprocessor(Dict(), Dict(), Dict(), nothing, 0.0, Float32[])
end

Preprocessor

In [17]:
# Assicurati di avere questa funzione helper definita prima
function assignment_costs(model, X)
    centers = model.centers
    dists = Float32[]
    for i in 1:size(X, 2)
        pt = X[:, i]
        d = minimum([norm(pt - centers[:, c]) for c in 1:size(centers, 2)])
        push!(dists, d)
    end
    return dists
end

function fit_transform!(preproc::Preprocessor, df::DataFrame)
    df_processed = copy(df)
    
    # 1. Date Parsing
    try
        df_processed[!, :TransactionDateObj] = DateTime.(df_processed[!, "Transaction Date"], "yyyy-mm-dd HH:MM:SS")
    catch
         df_processed[!, :TransactionDateObj] = Now() 
    end
    df_processed[!, :Hour] = hour.(df_processed[!, :TransactionDateObj])
    
    # 2. Engineering
    df_processed[!, :Velocity] = calculate_velocity(df_processed)
    df_processed[!, :AddrMismatch] = check_address_mismatch(df_processed)
    
    # 3. Scaling
    num_cols = ["Transaction Amount", "Customer Age", "Account Age Days", "Quantity", "Velocity"]
    for col_name in num_cols
        col = Symbol(col_name)
        data = Float32.(df_processed[!, col])
        mu = mean(data)
        sigma = std(data)
        preproc.means[col] = mu
        preproc.stds[col] = sigma
        df_processed[!, col] = (data .- mu) ./ (sigma + 1e-6)
    end
    
    # 4. Encoding
    cat_cols = ["Payment Method", "Product Category", "Device Used"]
    encoded_features = Matrix{Float32}(undef, 0, nrow(df_processed))
    for col_name in cat_cols
        col = Symbol(col_name)
        unique_vals = unique(df_processed[!, col])
        mapping = Dict(val => i for (i, val) in enumerate(unique_vals))
        preproc.cat_maps[col] = mapping
        indices = [get(mapping, x, 0) for x in df_processed[!, col]]
        oh = OneHotArrays.onehotbatch(indices, 1:length(unique_vals))
        encoded_features = vcat(encoded_features, Float32.(oh))
    end
    
    # 5. Assemblaggio
    numeric_data = Matrix{Float32}(df_processed[!, [Symbol(c) for c in num_cols]])'
    sin_h, cos_h = encode_cyclic(df_processed, :Hour, 24)
    cyclic_data = vcat(sin_h', cos_h')
    mismatch_data = Float32.(df_processed[!, :AddrMismatch])'
    
    X = vcat(numeric_data, cyclic_data, mismatch_data, encoded_features)
    
    # --- FIX CLUSTERING (maxiter) ---
    legit_indices = df_processed[!, "Is Fraudulent"] .== 0
    X_legit = X[:, legit_indices]
    
    # NOTA: Qui usiamo maxiter invece di max_iter
    R = kmeans(X_legit, 5; maxiter=50) 
    preproc.cluster_model = R
    
    distances_legit = assignment_costs(R, X_legit)
    preproc.cluster_threshold = quantile(distances_legit, 0.95)
    
    # --- TARGET ---
    original_labels = df_processed[!, "Is Fraudulent"]
    new_labels = zeros(Int, length(original_labels))
    all_distances = assignment_costs(preproc.cluster_model, X)
    
    for i in 1:length(original_labels)
        if original_labels[i] == 1
            new_labels[i] = 3
        elseif all_distances[i] > preproc.cluster_threshold
            new_labels[i] = 2
        else
            new_labels[i] = 1
        end
    end
    
    # Pesi
    counts = [count(x->x==i, new_labels) for i in 1:3]
    weights = Float32.(sum(counts) ./ (counts .+ 1))
    preproc.fraud_ratio = weights ./ maximum(weights)
    
    return X, new_labels
end

fit_transform! (generic function with 1 method)

In [18]:
function build_model(input_dim, output_dim)
    return Chain(
        Dense(input_dim => 64, relu),
        Dropout(0.3),
        Dense(64 => 32, relu),
        Dropout(0.2),
        Dense(32 => output_dim),
        softmax
    )
end

build_model (generic function with 1 method)

In [19]:
# --- CARICAMENTO DATI ---
# Sostituisci con il path corretto
df = CSV.read("Fraudulent_E-Commerce_Transaction_Data_merge.csv", DataFrame)

# --- SPLIT INIZIALE ---
# Importante: Split Train/Test PRIMA di tutto per evitare Data Leakage
train_idx, test_idx = splitobs(shuffleobs(1:nrow(df)), at=0.8)
df_train = df[train_idx, :]
df_test = df[test_idx, :]

println("Training samples: ", nrow(df_train))
println("Test samples: ", nrow(df_test))

# --- PIPELINE ---
# ... (dopo aver caricato df e fatto split) ...

preprocessor = Preprocessor()
println("Processing Training Set...")
X_train_raw, y_train_idx = fit_transform!(preprocessor, df_train)

# --- CORREZIONE IMPORTANTE: Casting a Float32 ---
X_train = Float32.(X_train_raw) # Converte tutto in Float32
y_train = OneHotArrays.onehotbatch(y_train_idx, 1:3)

println("Processing Test Set...")
X_test_raw, y_test_idx = transform_test(preprocessor, df_test)
X_test = Float32.(X_test_raw)   # Converte tutto in Float32
y_test = OneHotArrays.onehotbatch(y_test_idx, 1:3)

# ... (Ora puoi procedere con model = build_model(...) e il loop di training) ...

# --- TRAINING NEURAL NETWORK ---
input_dim = size(X_train, 1)
output_dim = 3
model = build_model(input_dim, output_dim)

# Setup Optimizer & Loss
opt = Flux.setup(Flux.Adam(0.001), model)
# Alpha calcolato automaticamente nel preprocessor
alpha_weights = reshape(preprocessor.fraud_ratio, 3, 1) 

println("Alpha Weights per Focal Loss: ", alpha_weights)

# DataLoader
train_data = Flux.DataLoader((X_train, y_train), batchsize=128, shuffle=true)

epochs = 20
println("\nInizio Training...")

for epoch in 1:epochs
    loss_sum = 0.0f0
    for (x, y) in train_data
        grads = Flux.gradient(model) do m
            y_hat = m(x)
            focal_loss(y_hat, y; alpha=alpha_weights)
        end
        Flux.update!(opt, model, grads[1])
        
        # Calcolo loss per display
        y_hat_curr = model(x)
        loss_sum += focal_loss(y_hat_curr, y; alpha=alpha_weights)
    end
    
    if epoch % 5 == 0
        @printf("Epoch %d: Loss = %.4f\n", epoch, loss_sum/length(train_data))
    end
end

# --- VALUTAZIONE ---
println("\n--- Valutazione Finale ---")

y_pred_probs = model(X_test)
y_pred_indices = onecold(y_pred_probs, 1:3)
y_true_indices = onecold(y_test, 1:3)

# Matrice di Confusione 3x3
conf_matrix = zeros(Int, 3, 3)
for i in 1:length(y_pred_indices)
    p = y_pred_indices[i]
    t = y_true_indices[i]
    conf_matrix[t, p] += 1
end

classes = ["Legit", "Suspicious", "Fraud"]

println("\nConfusion Matrix (Rows=True, Cols=Pred):")
display(conf_matrix)

# Calcolo Metriche per Classe
println("\nDettaglio Metriche:")
for i in 1:3
    tp = conf_matrix[i, i]
    fn = sum(conf_matrix[i, :]) - tp
    fp = sum(conf_matrix[:, i]) - tp
    tn = sum(conf_matrix) - (tp + fp + fn)
    
    sensitivity = tp / (tp + fn) # Recall
    precision = tp / (tp + fp)
    specificity = tn / (tn + fp)
    f1 = 2 * (precision * sensitivity) / (precision + sensitivity)
    
    # Gestione NaN se 0
    sensitivity = isnan(sensitivity) ? 0.0 : sensitivity
    precision = isnan(precision) ? 0.0 : precision
    f1 = isnan(f1) ? 0.0 : f1
    
    @printf("\nClasse %s:\n", classes[i])
    @printf("  Sensitivity (Recall): %.2f%%\n", sensitivity * 100)
    @printf("  Precision:            %.2f%%\n", precision * 100)
    @printf("  Specificity:          %.2f%%\n", specificity * 100)
    @printf("  F1 Score:             %.2f%%\n", f1 * 100)
end

println("\nAnalisi conclusa.")

Training samples: 1197269
Test samples: 299317
Processing Training Set...
Processing Test Set...
Alpha Weights per Focal Loss: Float32[0.05263304; 1.0; 0.948384;;]

Inizio Training...
Epoch 5: Loss = 0.0205
Epoch 10: Loss = 0.0200
Epoch 15: Loss = 0.0199
Epoch 20: Loss = 0.0198

--- Valutazione Finale ---

Confusion Matrix (Rows=True, Cols=Pred):


3×3 Matrix{Int64}:
 241391   6309  22523
     11  13745    239
   6246   2353   6500


Dettaglio Metriche:

Classe Legit:
  Sensitivity (Recall): 89.33%
  Precision:            97.47%
  Specificity:          78.49%
  F1 Score:             93.22%

Classe Suspicious:
  Sensitivity (Recall): 98.21%
  Precision:            61.34%
  Specificity:          96.96%
  F1 Score:             75.52%

Classe Fraud:
  Sensitivity (Recall): 43.05%
  Precision:            22.21%
  Specificity:          91.99%
  F1 Score:             29.31%

Analisi conclusa.
